In [1]:
BACKGROUND_VALUE = 220.0   # try 220 for light-gray right-image style


## Step 1. Handcrafted-feature regression baseline


### 1. Imports

In [2]:

import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### 2. Reproducibility


In [3]:
def set_seed(seed=5527):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(5527)


### 3. Load cleaned metadata file

In [4]:
final_df = pd.read_csv("county_image_and_poverty_rate_graybackground.csv", dtype={"county_id": str})

# Keep only what we need for the baseline
final_df = final_df[["image_path", "poverty_rate"]].copy()
final_df["poverty_rate"] = final_df["poverty_rate"].astype(float)

# Keep only PNG rows
final_df = final_df[final_df["image_path"].str.lower().str.endswith(".png")].copy()

print("Final dataframe shape:", final_df.shape)
print(final_df.head())
print(final_df["poverty_rate"].describe())

Final dataframe shape: (3109, 2)
                                          image_path  poverty_rate
0  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...          12.9
1  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...          10.5
2  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...          28.1
3  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...          21.6
4  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...          11.7
count    3109.000000
mean       14.110872
std         5.421440
min         3.800000
25%        10.400000
50%        13.200000
75%        16.800000
max        55.700000
Name: poverty_rate, dtype: float64


### 4. Train / validation / test split

In [5]:
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=5527
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=5527
)

print("\nSplit sizes")
print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)


Split sizes
Train: (2176, 2)
Val:   (466, 2)
Test:  (467, 2)


### 5. Load grayscale image + county mask


In [6]:
def load_gray_image(path, target_size=(64, 64), background_value=220.0):
    """
    Load a PNG as grayscale and composite transparent outside-county area
    onto a constant background.

    background_value:
        0.0   -> black background (left-image style)
        220.0 -> light gray background (right-image style)
        255.0 -> white background
    """
    path = str(path)
    suffix = Path(path).suffix.lower()

    if suffix != ".png":
        raise ValueError(f"Expected PNG file, got {suffix} for {path}")

    pil_img = Image.open(path).convert("RGBA")
    arr = np.array(pil_img, dtype=np.float32)   # (H, W, 4)

    rgb = arr[..., :3]
    alpha = arr[..., 3] / 255.0   # scale to [0,1]

    # grayscale intensity from RGB
    gray = rgb.mean(axis=-1)

    # composite onto chosen background
    img = alpha * gray + (1.0 - alpha) * background_value

    # clean image
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    img = np.clip(img, a_min=0, a_max=None)

    # helpful for skewed nighttime-light values
    img = np.log1p(img)

    # resize + pad to 64x64
    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

    _, _, h, w = img_tensor.shape
    target_h, target_w = target_size

    scale = min(target_h / h, target_w / w)
    new_h = max(1, int(round(h * scale)))
    new_w = max(1, int(round(w * scale)))

    img_tensor = F.interpolate(
        img_tensor,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    pad_h = target_h - new_h
    pad_w = target_w - new_w

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    img_tensor = F.pad(
        img_tensor,
        (pad_left, pad_right, pad_top, pad_bottom),
        mode="constant",
        value=np.log1p(background_value)
    )

    img = img_tensor.squeeze(0).squeeze(0).numpy().astype(np.float32)
    return img

In [7]:
sample_img = load_gray_image(train_df.iloc[0]["image_path"], background_value=220.0)
print(sample_img.shape)

(64, 64)


### 6. Handcrafted feature extraction

In [8]:
def extract_handcrafted_features(path, lit_threshold=0.10, bright_threshold=0.50):
    """
    Extract simple interpretable features from the full county image.
    No mask is used.
    """
    img = load_gray_image(path)
    pixels = img.reshape(-1)

    feats = {
        "mean_brightness": pixels.mean(),
        "max_brightness": pixels.max(),
        "std_brightness": pixels.std(),
        "median_brightness": np.median(pixels),
        "p25_brightness": np.percentile(pixels, 25),
        "p75_brightness": np.percentile(pixels, 75),
        "p90_brightness": np.percentile(pixels, 90),
        "p95_brightness": np.percentile(pixels, 95),

        "lit_pixel_share": np.mean(pixels > lit_threshold),
        "bright_pixel_share": np.mean(pixels > bright_threshold),
        "zero_pixel_share": np.mean(pixels <= 1e-6),
        "nonzero_pixel_share": np.mean(pixels > 1e-6)
    }

    return feats


# Optional: test one row
test_feats = extract_handcrafted_features(train_df.iloc[0]["image_path"])
print("\nSample handcrafted features:")
print(test_feats)


Sample handcrafted features:
{'mean_brightness': np.float32(2.4338603), 'max_brightness': np.float32(5.398163), 'std_brightness': np.float32(2.2553916), 'median_brightness': np.float32(3.1396146), 'p25_brightness': np.float32(0.0), 'p75_brightness': np.float32(4.487931), 'p90_brightness': np.float32(5.398163), 'p95_brightness': np.float32(5.398163), 'lit_pixel_share': np.float64(0.599609375), 'bright_pixel_share': np.float64(0.5810546875), 'zero_pixel_share': np.float64(0.394287109375), 'nonzero_pixel_share': np.float64(0.605712890625)}


### 7. Build feature tables


In [9]:
def build_feature_df(df):
    feature_rows = []

    for _, row in df.iterrows():
        feats = extract_handcrafted_features(row["image_path"])
        feats["image_path"] = row["image_path"]
        feats["poverty_rate"] = row["poverty_rate"]
        feature_rows.append(feats)

    return pd.DataFrame(feature_rows)


train_feat_df = build_feature_df(train_df)
val_feat_df = build_feature_df(val_df)
test_feat_df = build_feature_df(test_df)

print("\nTrain feature table shape:", train_feat_df.shape)
print(train_feat_df.head())


Train feature table shape: (2176, 14)
   mean_brightness  max_brightness  std_brightness  median_brightness  \
0         2.433860        5.398163        2.255392           3.139615   
1         2.694323        5.398163        2.292256           3.470545   
2         1.984943        5.398163        2.411955           0.000000   
3         0.541397        5.398163        1.592593           0.000000   
4         3.038077        5.523515        1.996897           3.785506   

   p25_brightness  p75_brightness  p90_brightness  p95_brightness  \
0        0.000000        4.487931        5.398163        5.398163   
1        0.000000        5.398163        5.398163        5.398163   
2        0.000000        5.398163        5.398163        5.398163   
3        0.000000        0.000000        2.759099        5.398163   
4        0.291979        4.565535        5.398163        5.398163   

   lit_pixel_share  bright_pixel_share  zero_pixel_share  nonzero_pixel_share  \
0         0.599609        

### 8. Prepare X and y

In [13]:
feature_cols = [
    "mean_brightness",
    "max_brightness",
    "std_brightness",
    "median_brightness",
    "p25_brightness",
    "p75_brightness",
    "p90_brightness",
    "p95_brightness",
    "lit_pixel_share",
    "bright_pixel_share",
    "zero_pixel_share",
    "nonzero_pixel_share"
]

X_train = train_feat_df[feature_cols].copy()
y_train = train_feat_df["poverty_rate"].copy()

X_val = val_feat_df[feature_cols].copy()
y_val = val_feat_df["poverty_rate"].copy()

X_test = test_feat_df[feature_cols].copy()
y_test = test_feat_df["poverty_rate"].copy()


### 9. Evaluation helper

In [14]:
def regression_metrics(y_true, y_pred, name="Model"):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R2:   {r2:.4f}")

    return {"rmse": rmse, "mae": mae, "r2": r2}

### 10. Ridge regression baseline

In [15]:
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

val_pred_ridge = ridge_model.predict(X_val)
test_pred_ridge = ridge_model.predict(X_test)

ridge_val_metrics = regression_metrics(y_val, val_pred_ridge, name="Ridge Validation")
ridge_test_metrics = regression_metrics(y_test, test_pred_ridge, name="Ridge Test")


Ridge Validation
RMSE: 5.4366
MAE:  4.1649
R2:   0.0146

Ridge Test
RMSE: 5.3855
MAE:  4.1309
R2:   0.0195


### 11. Random Forest baseline

In [16]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    random_state=5527,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

val_pred_rf = rf_model.predict(X_val)
test_pred_rf = rf_model.predict(X_test)

rf_val_metrics = regression_metrics(y_val, val_pred_rf, name="Random Forest Validation")
rf_test_metrics = regression_metrics(y_test, test_pred_rf, name="Random Forest Test")


Random Forest Validation
RMSE: 5.6405
MAE:  4.2947
R2:   -0.0607

Random Forest Test
RMSE: 5.4925
MAE:  4.1955
R2:   -0.0198


### 12. Compare results

In [17]:
results_df = pd.DataFrame([
    {
        "model": "Ridge",
        "val_rmse": ridge_val_metrics["rmse"],
        "val_mae": ridge_val_metrics["mae"],
        "val_r2": ridge_val_metrics["r2"],
        "test_rmse": ridge_test_metrics["rmse"],
        "test_mae": ridge_test_metrics["mae"],
        "test_r2": ridge_test_metrics["r2"],
    },
    {
        "model": "Random Forest",
        "val_rmse": rf_val_metrics["rmse"],
        "val_mae": rf_val_metrics["mae"],
        "val_r2": rf_val_metrics["r2"],
        "test_rmse": rf_test_metrics["rmse"],
        "test_mae": rf_test_metrics["mae"],
        "test_r2": rf_test_metrics["r2"],
    }
])

print("\nModel comparison")
print(results_df)


Model comparison
           model  val_rmse   val_mae    val_r2  test_rmse  test_mae   test_r2
0          Ridge  5.436565  4.164906  0.014581   5.385526  4.130867  0.019538
1  Random Forest  5.640518  4.294664 -0.060742   5.492468  4.195457 -0.019787


### 13. Random Forest feature importance

In [18]:
rf_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nRandom Forest feature importance")
print(rf_importance_df)


Random Forest feature importance
                feature  importance
2        std_brightness    0.221187
0       mean_brightness    0.181354
9    bright_pixel_share    0.111713
3     median_brightness    0.096383
8       lit_pixel_share    0.079701
5        p75_brightness    0.077998
10     zero_pixel_share    0.073777
11  nonzero_pixel_share    0.060044
1        max_brightness    0.034134
6        p90_brightness    0.031729
7        p95_brightness    0.021394
4        p25_brightness    0.010586




""" 14. Save outputs """
train_feat_df.to_csv("train_handcrafted_features.csv", index=False)
val_feat_df.to_csv("val_handcrafted_features.csv", index=False)
test_feat_df.to_csv("test_handcrafted_features.csv", index=False)
results_df.to_csv("handcrafted_baseline_results.csv", index=False)
rf_importance_df.to_csv("rf_feature_importance.csv", index=False)

print("\nSaved:")
print("- train_handcrafted_features.csv")
print("- val_handcrafted_features.csv")
print("- test_handcrafted_features.csv")
print("- handcrafted_baseline_results.csv")
print("- rf_feature_importance.csv")

"""

## Step 2. Hybrid model: CNN embeddings + Ridge/Random Forest

### 1. 1-channel image pipeline

In [19]:
from PIL import Image
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path

def load_image(path, target_size=(64, 64)):
    path = str(path)
    suffix = Path(path).suffix.lower()

    if suffix != ".png":
        raise ValueError(f"Expected PNG file, got {suffix} for {path}")

    pil_img = Image.open(path).convert("RGBA")
    arr = np.array(pil_img, dtype=np.float32)

    # grayscale from RGB
    rgb = arr[..., :3]
    img = rgb.mean(axis=-1)

    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    img = np.clip(img, a_min=0, a_max=None)
    img = np.log1p(img)

    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    _, _, h, w = img_tensor.shape
    target_h, target_w = target_size

    scale = min(target_h / h, target_w / w)
    new_h = max(1, int(round(h * scale)))
    new_w = max(1, int(round(w * scale)))

    img_tensor = F.interpolate(
        img_tensor,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    pad_h = target_h - new_h
    pad_w = target_w - new_w

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    img_tensor = F.pad(
        img_tensor,
        (pad_left, pad_right, pad_top, pad_bottom),
        mode="constant",
        value=0.0
    )

    img = img_tensor.squeeze(0).squeeze(0).numpy()
    return img.astype(np.float32)

### 2. Compute image normalization stats

In [21]:

y_train_mean = train_df["poverty_rate"].mean()
y_train_std = train_df["poverty_rate"].std()

print("y_train_mean:", y_train_mean)
print("y_train_std:", y_train_std)


y_train_mean: 14.12527573529412
y_train_std: 5.405674746427598


In [22]:
def compute_mean_std(df):
    total_sum = 0.0
    total_sq_sum = 0.0
    total_pixels = 0

    for path in df["image_path"]:
        img = load_image(path)
        total_sum += img.sum()
        total_sq_sum += (img ** 2).sum()
        total_pixels += img.size

    mean = total_sum / total_pixels
    var = total_sq_sum / total_pixels - mean ** 2
    std = np.sqrt(max(var, 1e-8))

    return float(mean), float(std)

train_mean, train_std = compute_mean_std(train_df)
print("Train mean:", train_mean)
print("Train std: ", train_std)

Train mean: 0.7130646109580994
Train std:  1.4683489799499512


### 3. 1-channel dataset

In [23]:
from torch.utils.data import Dataset, DataLoader

class CountyPovertyDataset1C(Dataset):
    def __init__(self, df, img_mean, img_std, y_mean, y_std):
        self.df = df.reset_index(drop=True).copy()
        self.img_mean = img_mean
        self.img_std = img_std
        self.y_mean = y_mean
        self.y_std = y_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = load_image(row["image_path"])
        img = (img - self.img_mean) / (self.img_std + 1e-6)
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)  # (1,64,64)

        y = float(row["poverty_rate"])
        y = (y - self.y_mean) / (self.y_std + 1e-6)
        y = torch.tensor(y, dtype=torch.float32)

        return img, y

In [24]:
batch_size = 32

train_dataset = CountyPovertyDataset1C(train_df, train_mean, train_std, y_train_mean, y_train_std)
val_dataset = CountyPovertyDataset1C(val_df, train_mean, train_std, y_train_mean, y_train_std)
test_dataset = CountyPovertyDataset1C(test_df, train_mean, train_std, y_train_mean, y_train_std)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

### 4. CNN embeddings

In [25]:
import torch.nn as nn

class CNNRegressor1C(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 -> 32

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 -> 16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x, return_embedding=False):
        x = self.features(x)
        embedding = torch.flatten(x, start_dim=1)   # shape [batch, 64]
        out = self.regressor(x).squeeze(1)

        if return_embedding:
            return out, embedding
        return out

### 5. Train the 1-channel CNN model

In [26]:
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = CNNRegressor1C().to(device)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

Using device: cpu


In [27]:
def evaluate_regression(model, loader, criterion, device, y_mean, y_std):
    model.eval()

    all_preds_std = []
    all_targets_std = []
    running_loss = 0.0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)
            loss = criterion(preds, targets)

            running_loss += loss.item() * images.size(0)

            all_preds_std.extend(preds.cpu().numpy())
            all_targets_std.extend(targets.cpu().numpy())

    all_preds_std = np.array(all_preds_std)
    all_targets_std = np.array(all_targets_std)

    all_preds = all_preds_std * (y_std + 1e-6) + y_mean
    all_targets = all_targets_std * (y_std + 1e-6) + y_mean

    mse = mean_squared_error(all_targets, all_preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)

    avg_loss = running_loss / len(loader.dataset)

    return {
        "loss": avg_loss,
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

In [28]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device,
                y_mean, y_std, epochs=25, patience=10):
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_rmse": [],
        "val_mae": [],
        "val_r2": []
    }

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate_regression(model, val_loader, criterion, device, y_mean, y_std)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_rmse"].append(val_metrics["rmse"])
        history["val_mae"].append(val_metrics["mae"])
        history["val_r2"].append(val_metrics["r2"])

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE: {val_metrics['rmse']:.4f} | "
            f"Val MAE: {val_metrics['mae']:.4f} | "
            f"Val R2: {val_metrics['r2']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

In [29]:
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    y_mean=y_train_mean,
    y_std=y_train_std,
    epochs=10,
    patience=10
)

Epoch 01 | Train Loss: 0.3936 | Val Loss: 0.3931 | Val RMSE: 5.5346 | Val MAE: 4.0393 | Val R2: -0.0213
Epoch 02 | Train Loss: 0.3766 | Val Loss: 0.3898 | Val RMSE: 5.3951 | Val MAE: 4.0810 | Val R2: 0.0296
Epoch 03 | Train Loss: 0.3726 | Val Loss: 0.3946 | Val RMSE: 5.3492 | Val MAE: 4.1631 | Val R2: 0.0460
Epoch 04 | Train Loss: 0.3726 | Val Loss: 0.3847 | Val RMSE: 5.4640 | Val MAE: 3.9660 | Val R2: 0.0046
Epoch 05 | Train Loss: 0.3694 | Val Loss: 0.3983 | Val RMSE: 5.5987 | Val MAE: 4.0215 | Val R2: -0.0451
Epoch 06 | Train Loss: 0.3650 | Val Loss: 0.3941 | Val RMSE: 5.3374 | Val MAE: 4.1680 | Val R2: 0.0502
Epoch 07 | Train Loss: 0.3661 | Val Loss: 0.3849 | Val RMSE: 5.4559 | Val MAE: 3.9812 | Val R2: 0.0076
Epoch 08 | Train Loss: 0.3636 | Val Loss: 0.3830 | Val RMSE: 5.3116 | Val MAE: 4.0476 | Val R2: 0.0594
Epoch 09 | Train Loss: 0.3629 | Val Loss: 0.3832 | Val RMSE: 5.4300 | Val MAE: 3.9589 | Val R2: 0.0170
Epoch 10 | Train Loss: 0.3624 | Val Loss: 0.3771 | Val RMSE: 5.3212 | V

In [30]:
test_metrics_cnn = evaluate_regression(
    model,
    test_loader,
    criterion,
    device,
    y_mean=y_train_mean,
    y_std=y_train_std
)

print("\nEnd-to-end CNN Test")
print(f"RMSE: {test_metrics_cnn['rmse']:.4f}")
print(f"MAE:  {test_metrics_cnn['mae']:.4f}")
print(f"R2:   {test_metrics_cnn['r2']:.4f}")


End-to-end CNN Test
RMSE: 5.2548
MAE:  3.9337
R2:   0.0665


### 6. Extract embeddings

In [31]:
def extract_embeddings(model, loader, device, y_mean, y_std):
    model.eval()

    all_embeddings = []
    all_targets_std = []
    all_preds_std = []

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)

            preds, embeddings = model(images, return_embedding=True)

            all_embeddings.append(embeddings.cpu().numpy())
            all_targets_std.extend(targets.numpy())
            all_preds_std.extend(preds.cpu().numpy())

    X = np.vstack(all_embeddings)
    y_std_arr = np.array(all_targets_std)
    y = y_std_arr * (y_std + 1e-6) + y_mean

    cnn_preds = np.array(all_preds_std) * (y_std + 1e-6) + y_mean

    return X, y, cnn_preds

In [32]:
X_train_emb, y_train_emb, train_cnn_pred = extract_embeddings(model, train_loader, device, y_train_mean, y_train_std)
X_val_emb, y_val_emb, val_cnn_pred = extract_embeddings(model, val_loader, device, y_train_mean, y_train_std)
X_test_emb, y_test_emb, test_cnn_pred = extract_embeddings(model, test_loader, device, y_train_mean, y_train_std)

print("Embedding train shape:", X_train_emb.shape)
print("Embedding val shape:  ", X_val_emb.shape)
print("Embedding test shape: ", X_test_emb.shape)

Embedding train shape: (2176, 64)
Embedding val shape:   (466, 64)
Embedding test shape:  (467, 64)


### 7. Fit Ridge and Random Forest on CNN embeddings

In [33]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

def regression_metrics(y_true, y_pred, name="Model"):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R2:   {r2:.4f}")

    return {"rmse": rmse, "mae": mae, "r2": r2}

Ridge on embeddings

In [34]:
ridge_emb_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_emb_model.fit(X_train_emb, y_train_emb)

val_pred_ridge_emb = ridge_emb_model.predict(X_val_emb)
test_pred_ridge_emb = ridge_emb_model.predict(X_test_emb)

ridge_emb_val_metrics = regression_metrics(y_val_emb, val_pred_ridge_emb, name="Ridge on CNN Embeddings Validation")
ridge_emb_test_metrics = regression_metrics(y_test_emb, test_pred_ridge_emb, name="Ridge on CNN Embeddings Test")


Ridge on CNN Embeddings Validation
RMSE: 5.4338
MAE:  4.1157
R2:   0.0156

Ridge on CNN Embeddings Test
RMSE: 5.1612
MAE:  3.8722
R2:   0.0995


Random Forest on embeddings

In [35]:
rf_emb_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    random_state=5527,
    n_jobs=-1
)

rf_emb_model.fit(X_train_emb, y_train_emb)

val_pred_rf_emb = rf_emb_model.predict(X_val_emb)
test_pred_rf_emb = rf_emb_model.predict(X_test_emb)

rf_emb_val_metrics = regression_metrics(y_val_emb, val_pred_rf_emb, name="RF on CNN Embeddings Validation")
rf_emb_test_metrics = regression_metrics(y_test_emb, test_pred_rf_emb, name="RF on CNN Embeddings Test")


RF on CNN Embeddings Validation
RMSE: 5.2772
MAE:  4.0156
R2:   0.0715

RF on CNN Embeddings Test
RMSE: 5.1180
MAE:  3.8587
R2:   0.1145


In [36]:
hybrid_results_df = pd.DataFrame([
    {
        "model": "End_to_end_CNN",
        "test_rmse": test_metrics_cnn["rmse"],
        "test_mae": test_metrics_cnn["mae"],
        "test_r2": test_metrics_cnn["r2"],
    },
    {
        "model": "Ridge_on_CNN_Embeddings",
        "test_rmse": ridge_emb_test_metrics["rmse"],
        "test_mae": ridge_emb_test_metrics["mae"],
        "test_r2": ridge_emb_test_metrics["r2"],
    },
    {
        "model": "RF_on_CNN_Embeddings",
        "test_rmse": rf_emb_test_metrics["rmse"],
        "test_mae": rf_emb_test_metrics["mae"],
        "test_r2": rf_emb_test_metrics["r2"],
    }
])

print("\nHybrid comparison")
print(hybrid_results_df)

hybrid_results_df.to_csv("hybrid_results_nomask.csv", index=False)


Hybrid comparison
                     model  test_rmse  test_mae   test_r2
0           End_to_end_CNN   5.254846  3.933670  0.066543
1  Ridge_on_CNN_Embeddings   5.161226  3.872166  0.099508
2     RF_on_CNN_Embeddings   5.117990  3.858699  0.114531
